**1-Installation PySpark**

In [ ]:
!pip install pyspark

**2-Création le dataset dans Colab**

In [ ]:
import pandas as pd

data = {
    "HeuresEtude": [2,3,4,5,6,7,8,9,5,4],
    "Absences": [10,8,6,5,4,3,2,1,7,8],
    "Participation": [4,5,6,7,8,8,9,10,5,4],
    "Note": [8,10,11,13,15,16,18,19,12,10]
}

df = pd.DataFrame(data)

df.to_csv("students.csv", index=False)

print(df)

   HeuresEtude  Absences  Participation  Note
0            2        10              4     8
1            3         8              5    10
2            4         6              6    11
3            5         5              7    13
4            6         4              8    15
5            7         3              8    16
6            8         2              9    18
7            9         1             10    19
8            5         7              5    12
9            4         8              4    10


3-Création de la Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StudentPrediction") \
    .getOrCreate()

print("Spark démarré avec succès")

Spark démarré avec succès


**4-Charger le fichier CSV avec Spark**

In [ ]:
students = spark.read.csv(
    "students.csv",
    header=True,
    inferSchema=True
)

students.show()

+-----------+--------+-------------+----+
|HeuresEtude|Absences|Participation|Note|
+-----------+--------+-------------+----+
|          2|      10|            4|   8|
|          3|       8|            5|  10|
|          4|       6|            6|  11|
|          5|       5|            7|  13|
|          6|       4|            8|  15|
|          7|       3|            8|  16|
|          8|       2|            9|  18|
|          9|       1|           10|  19|
|          5|       7|            5|  12|
|          4|       8|            4|  10|
+-----------+--------+-------------+----+



**5-Explorer les données**

In [ ]:
students.printSchema()

root
 |-- HeuresEtude: integer (nullable = true)
 |-- Absences: integer (nullable = true)
 |-- Participation: integer (nullable = true)
 |-- Note: integer (nullable = true)



In [ ]:
students.describe().show()

+-------+------------------+-----------------+------------------+-----------------+
|summary|       HeuresEtude|         Absences|     Participation|             Note|
+-------+------------------+-----------------+------------------+-----------------+
|  count|                10|               10|                10|               10|
|   mean|               5.3|              5.4|               6.6|             13.2|
| stddev|2.2135943621178655|2.913569784454954|2.1186998109427604|3.675746333890726|
|    min|                 2|                1|                 4|                8|
|    max|                 9|               10|                10|               19|
+-------+------------------+-----------------+------------------+-----------------+



**6-Créer la colonne features**

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["HeuresEtude", "Absences", "Participation"],
    outputCol="features"
)

data = assembler.transform(students)

data.select("features", "Note").show(truncate=False)

+--------------+----+
|features      |Note|
+--------------+----+
|[2.0,10.0,4.0]|8   |
|[3.0,8.0,5.0] |10  |
|[4.0,6.0,6.0] |11  |
|[5.0,5.0,7.0] |13  |
|[6.0,4.0,8.0] |15  |
|[7.0,3.0,8.0] |16  |
|[8.0,2.0,9.0] |18  |
|[9.0,1.0,10.0]|19  |
|[5.0,7.0,5.0] |12  |
|[4.0,8.0,4.0] |10  |
+--------------+----+



**7-Séparer les données en entraînement et test**

In [ ]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Nombre de lignes train :", train_data.count())
print("Nombre de lignes test :", test_data.count())

Nombre de lignes train : 7
Nombre de lignes test : 3


**8-Créer et entraîner le modèle**

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="Note"
)

model = lr.fit(train_data)

print("Modèle entraîné avec succès")

Modèle entraîné avec succès


**9-Faire les prédictions**

In [ ]:
predictions = model.transform(test_data)

predictions.select(
    "HeuresEtude",
    "Absences",
    "Participation",
    "Note",
    "prediction"
).show()

+-----------+--------+-------------+----+------------------+
|HeuresEtude|Absences|Participation|Note|        prediction|
+-----------+--------+-------------+----+------------------+
|          4|       6|            6|  11|11.555555555555493|
|          6|       4|            8|  15|14.777777777777786|
|          8|       2|            9|  18| 17.46464646464647|
+-----------+--------+-------------+----+------------------+



**10-Évaluer le modèle**

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="Note",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)

print("RMSE =", rmse)

RMSE = 0.46354722850840646


**11-Tester un nouvel étudiant**

In [ ]:
new_student = spark.createDataFrame(
    [(7, 2, 9)],
    ["HeuresEtude", "Absences", "Participation"]
)

new_student = assembler.transform(new_student)

result = model.transform(new_student)

result.select("prediction").show()

+-----------------+
|       prediction|
+-----------------+
|16.52525252525248|
+-----------------+

